In [20]:
# Installing the pyspark

!pip install pyspark --quiet

In [21]:
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, avg, count, when

# Creating the spark session
spark = SparkSession.builder \
      .appName('Day4_big_data_sales')\
      .getOrCreate()

## Understanding PySpark: Core Concepts

PySpark is the Python API for Apache Spark, an open-source, distributed computing system used for big data processing and analytics. It allows you to write Spark applications using Python.

### 1. SparkSession

`SparkSession` is the entry point to programming Spark with the Dataset and DataFrame API. It unifies all the different contexts (SQLContext, HiveContext, etc.) available in previous Spark versions. It's how you interact with Spark.

### 2. DataFrame

A `DataFrame` in Spark is a distributed collection of data organized into named columns. It's conceptually equivalent to a table in a relational database or a DataFrame in R/Python (Pandas), but with richer optimizations under the hood. Spark DataFrames are immutable, fault-tolerant, and can handle petabytes of data.

### 3. RDD (Resilient Distributed Dataset)

`RDD`s are the fundamental data structure of Spark. They are immutable, fault-tolerant, distributed collections of objects. While DataFrames are now the preferred API for most tasks due to their higher-level abstractions and optimizations, RDDs provide the lowest-level API and are useful for unstructured data or when you need fine-grained control over transformations.

### 4. Transformations

Transformations are operations on DataFrames (or RDDs) that create a new DataFrame from an existing one. They are *lazy*, meaning they don't execute immediately when called. Instead, they build a logical plan (or lineage graph) of computations. Examples include `filter()`, `select()`, `groupBy()`, `withColumn()`.

### 5. Actions

Actions are operations that trigger the execution of the lazy transformations defined earlier. They return a result to the driver program or write data to an external storage system. Examples include `show()`, `count()`, `collect()`, `write()`.


## Loading Data into a PySpark DataFrame

First, we need to load our `large_sales_data` dataset. Assuming it's a CSV file, we'll use `spark.read.csv()`.

In [22]:
# Load the dataset
sales_df = spark.read.csv("/content/large_sales_data.csv", header=True, inferSchema=True)

print("PySpark DataFrame loaded successfully!")

PySpark DataFrame loaded successfully!


In [23]:
# Create a dummy large_sales_data.csv file for demonstration
import pandas as pd
import numpy as np

np.random.seed(42)
data_size = 1000 # Small size for quick demo, but conceptually represents 'large'

dummy_data = {
    'TransactionID': np.arange(1, data_size + 1),
    'ProductID': np.random.randint(101, 110, data_size),
    'Category': np.random.choice(['Electronics', 'Clothing', 'Home Goods', 'Books'], data_size),
    'Quantity': np.random.randint(1, 10, data_size),
    'Price': np.round(np.random.uniform(10.0, 500.0, data_size), 2),
    'CustomerID': np.random.randint(1001, 1050, data_size),
    'TransactionDate': pd.to_datetime('2023-01-01') + pd.to_timedelta(np.random.randint(0, 365, data_size), unit='D'),
    'Region': np.random.choice(['North', 'South', 'East', 'West'], data_size)
}

dummy_df_pandas = pd.DataFrame(dummy_data)
dummy_df_pandas.to_csv('large_sales_data.csv', index=False)

print("Dummy 'large_sales_data.csv' created successfully!")

Dummy 'large_sales_data.csv' created successfully!


In [24]:
# Load the dataset (re-attempt after creating dummy file)
sales_df = spark.read.csv("large_sales_data.csv", header=True, inferSchema=True)

print("PySpark DataFrame loaded successfully!")



PySpark DataFrame loaded successfully!


## Exploratory Data Operations with PySpark DataFrames

Just like with Pandas, it's crucial to understand the structure and content of your data. PySpark DataFrames offer similar functionalities for this purpose, but optimized for distributed processing.

### 1. Displaying the Schema (`.printSchema()`)

The schema shows the column names and their inferred data types. This is essential for understanding how Spark has interpreted your data.

In [25]:
# Display the schema
sales_df.printSchema()

root
 |-- TransactionID: integer (nullable = true)
 |-- ProductID: integer (nullable = true)
 |-- Category: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Price: double (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- TransactionDate: date (nullable = true)
 |-- Region: string (nullable = true)



### 2. Viewing the First Few Rows (`.show()`)

The `.show()` action allows you to see the actual data in your DataFrame. By default, it shows the first 20 rows and truncates long strings.

In [26]:
# Show the first 5 rows of the DataFrame
sales_df.show(5)

+-------------+---------+-----------+--------+------+----------+---------------+------+
|TransactionID|ProductID|   Category|Quantity| Price|CustomerID|TransactionDate|Region|
+-------------+---------+-----------+--------+------+----------+---------------+------+
|            1|      107|      Books|       6|290.94|      1042|     2023-08-02|  East|
|            2|      104| Home Goods|       1|332.82|      1001|     2023-10-18| South|
|            3|      108| Home Goods|       9|133.63|      1029|     2023-01-22|  East|
|            4|      105| Home Goods|       9|117.91|      1034|     2023-07-12|  East|
|            5|      107|Electronics|       2| 249.3|      1013|     2023-05-30| South|
+-------------+---------+-----------+--------+------+----------+---------------+------+
only showing top 5 rows


### 3. Getting Descriptive Statistics (`.describe()`)

`.describe()` computes common summary statistics (count, mean, stddev, min, max) for numeric columns, and count, unique, top, freq for string columns. This helps in understanding the distribution and range of your data.

In [27]:
# Get descriptive statistics
sales_df.describe().show()

+-------+-----------------+------------------+----------+----------------+------------------+------------------+------+
|summary|    TransactionID|         ProductID|  Category|        Quantity|             Price|        CustomerID|Region|
+-------+-----------------+------------------+----------+----------------+------------------+------------------+------+
|  count|             1000|              1000|      1000|            1000|              1000|              1000|  1000|
|   mean|            500.5|           104.898|      NULL|           5.032|260.01010999999954|          1025.275|  NULL|
| stddev|288.8194360957494|2.6039155854935805|      NULL|2.55254608018964|140.52377799386673|14.097592600054679|  NULL|
|    min|                1|               101|     Books|               1|             10.22|              1001|  East|
|    max|             1000|               109|Home Goods|               9|            499.86|              1049|  West|
+-------+-----------------+-------------

### 4. Counting Rows (`.count()`)

To quickly get the total number of records in your DataFrame, use the `.count()` action.

In [28]:
# Count the number of rows
print(f"Total number of records: {sales_df.count()}")

Total number of records: 1000


## PySpark Data Transformations

PySpark DataFrames allow for powerful and efficient data manipulation through transformations. Remember, transformations are lazy operations; they build a plan and execute only when an action is called.

### 1. Filtering Data (`.filter()` / `.where()`)

Filtering allows you to select rows based on a specified condition. It's similar to the `WHERE` clause in SQL or boolean indexing in Pandas.

In [29]:
# Filter sales data for 'Electronics' category and Quantity > 5
electronics_sales_df = sales_df.filter((col("Category") == "Electronics") & (col("Quantity") > 5))

print("Filtered sales (Electronics, Quantity > 5):")
electronics_sales_df.show(5)

Filtered sales (Electronics, Quantity > 5):
+-------------+---------+-----------+--------+------+----------+---------------+------+
|TransactionID|ProductID|   Category|Quantity| Price|CustomerID|TransactionDate|Region|
+-------------+---------+-----------+--------+------+----------+---------------+------+
|           10|      104|Electronics|       8|  41.7|      1047|     2023-11-12| South|
|           13|      103|Electronics|       7|350.28|      1049|     2023-09-27|  East|
|           31|      103|Electronics|       8|143.07|      1043|     2023-08-25| North|
|           72|      107|Electronics|       7|465.42|      1047|     2023-07-07|  West|
|           80|      108|Electronics|       7| 155.4|      1002|     2023-09-20|  East|
+-------------+---------+-----------+--------+------+----------+---------------+------+
only showing top 5 rows


### 2. Selecting Columns (`.select()`)

`.select()` is used to choose specific columns from a DataFrame. You can also rename columns or create new ones during selection.

In [30]:
# Select specific columns: ProductID, Category, Quantity, Price
selected_columns_df = sales_df.select("ProductID", "Category", "Quantity", "Price")

print("Selected columns:")
selected_columns_df.show(5)

Selected columns:
+---------+-----------+--------+------+
|ProductID|   Category|Quantity| Price|
+---------+-----------+--------+------+
|      107|      Books|       6|290.94|
|      104| Home Goods|       1|332.82|
|      108| Home Goods|       9|133.63|
|      105| Home Goods|       9|117.91|
|      107|Electronics|       2| 249.3|
+---------+-----------+--------+------+
only showing top 5 rows


### 3. Grouping and Aggregating Data (`.groupBy()` and `.agg()`)

These operations are crucial for summarizing data. You can group data by one or more columns and then apply aggregate functions (like sum, average, count) to other columns within each group.

In [31]:
# Calculate total sales quantity and average price per category
aggregated_sales_df = sales_df.groupBy("Category").agg(
    sum("Quantity").alias("TotalQuantitySold"),
    avg("Price").alias("AveragePrice")
)

print("Aggregated sales by Category:")
aggregated_sales_df.show()

Aggregated sales by Category:
+-----------+-----------------+------------------+
|   Category|TotalQuantitySold|      AveragePrice|
+-----------+-----------------+------------------+
|Electronics|             1276| 249.5528346456693|
|   Clothing|             1349| 270.1353461538461|
|      Books|             1168|263.46075000000013|
| Home Goods|             1239| 256.7395121951221|
+-----------+-----------------+------------------+



### 4. Adding New Columns (`.withColumn()`)

`.withColumn()` allows you to add a new column or replace an existing one based on an expression.

In [32]:
# Add a new column 'TotalPrice' (Quantity * Price)
from pyspark.sql.functions import col

sales_with_total_price_df = sales_df.withColumn(
    "TotalPrice", col("Quantity") * col("Price")
)

print("DataFrame with 'TotalPrice' column:")
sales_with_total_price_df.show(5)

DataFrame with 'TotalPrice' column:
+-------------+---------+-----------+--------+------+----------+---------------+------+------------------+
|TransactionID|ProductID|   Category|Quantity| Price|CustomerID|TransactionDate|Region|        TotalPrice|
+-------------+---------+-----------+--------+------+----------+---------------+------+------------------+
|            1|      107|      Books|       6|290.94|      1042|     2023-08-02|  East|1745.6399999999999|
|            2|      104| Home Goods|       1|332.82|      1001|     2023-10-18| South|            332.82|
|            3|      108| Home Goods|       9|133.63|      1029|     2023-01-22|  East|           1202.67|
|            4|      105| Home Goods|       9|117.91|      1034|     2023-07-12|  East|           1061.19|
|            5|      107|Electronics|       2| 249.3|      1013|     2023-05-30| South|             498.6|
+-------------+---------+-----------+--------+------+----------+---------------+------+------------------+
o

## Side-by-Side Comparison: Pandas vs. PySpark

Let's perform the same operations we just did with PySpark using Pandas to highlight the syntax differences and conceptual approach.

### Preparation: Load Data into Pandas DataFrame

First, we'll load the same `large_sales_data.csv` into a Pandas DataFrame.

In [33]:
# Load the dataset into a Pandas DataFrame
pandas_sales_df = pd.read_csv("large_sales_data.csv")

print("Pandas DataFrame loaded successfully!")
print("Pandas DataFrame head:")
print(pandas_sales_df.head())

Pandas DataFrame loaded successfully!
Pandas DataFrame head:
   TransactionID  ProductID     Category  Quantity   Price  CustomerID  \
0              1        107        Books         6  290.94        1042   
1              2        104   Home Goods         1  332.82        1001   
2              3        108   Home Goods         9  133.63        1029   
3              4        105   Home Goods         9  117.91        1034   
4              5        107  Electronics         2  249.30        1013   

  TransactionDate Region  
0      2023-08-02   East  
1      2023-10-18  South  
2      2023-01-22   East  
3      2023-07-12   East  
4      2023-05-30  South  


### Comparison 1: Filtering Data

**PySpark:**
```python
electronics_sales_df = sales_df.filter((col("Category") == "Electronics") & (col("Quantity") > 5))
```

**Pandas:**

In [34]:
# Pandas equivalent for filtering
pandas_electronics_sales_df = pandas_sales_df[
    (pandas_sales_df["Category"] == "Electronics") &
    (pandas_sales_df["Quantity"] > 5)
]

print("Filtered sales (Pandas - Electronics, Quantity > 5):")
print(pandas_electronics_sales_df.head())

Filtered sales (Pandas - Electronics, Quantity > 5):
    TransactionID  ProductID     Category  Quantity   Price  CustomerID  \
9              10        104  Electronics         8   41.70        1047   
12             13        103  Electronics         7  350.28        1049   
30             31        103  Electronics         8  143.07        1043   
71             72        107  Electronics         7  465.42        1047   
79             80        108  Electronics         7  155.40        1002   

   TransactionDate Region  
9       2023-11-12  South  
12      2023-09-27   East  
30      2023-08-25  North  
71      2023-07-07   West  
79      2023-09-20   East  


### Comparison 2: Selecting Columns

**PySpark:**
```python
selected_columns_df = sales_df.select("ProductID", "Category", "Quantity", "Price")
```

**Pandas:**

In [35]:
# Pandas equivalent for selecting columns
pandas_selected_columns_df = pandas_sales_df[["ProductID", "Category", "Quantity", "Price"]]

print("Selected columns (Pandas):")
print(pandas_selected_columns_df.head())

Selected columns (Pandas):
   ProductID     Category  Quantity   Price
0        107        Books         6  290.94
1        104   Home Goods         1  332.82
2        108   Home Goods         9  133.63
3        105   Home Goods         9  117.91
4        107  Electronics         2  249.30


### Comparison 3: Grouping and Aggregating Data

**PySpark:**
```python
aggregated_sales_df = sales_df.groupBy("Category").agg(
    sum("Quantity").alias("TotalQuantitySold"),
    avg("Price").alias("AveragePrice")
)
```

**Pandas:**

In [36]:
# Pandas equivalent for grouping and aggregating
pandas_aggregated_sales_df = pandas_sales_df.groupby("Category").agg(
    TotalQuantitySold=('Quantity', 'sum'),
    AveragePrice=('Price', 'mean')
).reset_index()

print("Aggregated sales by Category (Pandas):")
print(pandas_aggregated_sales_df)

Aggregated sales by Category (Pandas):
      Category  TotalQuantitySold  AveragePrice
0        Books               1168    263.460750
1     Clothing               1349    270.135346
2  Electronics               1276    249.552835
3   Home Goods               1239    256.739512


### Comparison 4: Adding New Columns

**PySpark:**
```python
sales_with_total_price_df = sales_df.withColumn(
    "TotalPrice", col("Quantity") * col("Price")
)
```

**Pandas:**

In [37]:
# Pandas equivalent for adding a new column
pandas_sales_with_total_price_df = pandas_sales_df.copy()
pandas_sales_with_total_price_df["TotalPrice"] = pandas_sales_with_total_price_df["Quantity"] * pandas_sales_with_total_price_df["Price"]

print("DataFrame with 'TotalPrice' column (Pandas):")
print(pandas_sales_with_total_price_df.head())

DataFrame with 'TotalPrice' column (Pandas):
   TransactionID  ProductID     Category  Quantity   Price  CustomerID  \
0              1        107        Books         6  290.94        1042   
1              2        104   Home Goods         1  332.82        1001   
2              3        108   Home Goods         9  133.63        1029   
3              4        105   Home Goods         9  117.91        1034   
4              5        107  Electronics         2  249.30        1013   

  TransactionDate Region  TotalPrice  
0      2023-08-02   East     1745.64  
1      2023-10-18  South      332.82  
2      2023-01-22   East     1202.67  
3      2023-07-12   East     1061.19  
4      2023-05-30  South      498.60  


## Limitations of Pandas with Big Data and Why PySpark Succeeds

While Pandas is an incredibly powerful and user-friendly library for data manipulation in Python, it faces significant limitations when dealing with datasets that exceed the memory capacity of a single machine. This is where PySpark truly shines.

### Limitations of Pandas:

1.  **Memory Constraints (Single-Machine Execution):**
    *   Pandas DataFrames are *in-memory* objects. This means the entire dataset must fit into the RAM of the machine running the Pandas process. For datasets ranging from gigabytes to terabytes, this becomes a severe bottleneck, leading to `MemoryError` or extremely slow operations due to excessive swapping to disk.
    *   **Implication for Big Data:** You cannot process datasets larger than your machine's available RAM.

2.  **Lack of Parallelism and Distributed Computing:**
    *   Pandas operations are executed on a *single CPU core* by default. While some operations can leverage multiple cores on a single machine (e.g., using `apply` with multiprocessing), it's not inherently designed for distributed computation across a cluster of machines.
    *   **Implication for Big Data:** Processing large datasets sequentially on a single core is prohibitively slow, even if they fit in memory.

3.  **No Built-in Fault Tolerance:**
    *   If a Pandas operation fails midway through a long computation (e.g., due to an unexpected error or system crash), you typically lose all progress and have to restart from the beginning.

### Why PySpark Succeeds:

PySpark is built on Apache Spark, which was designed from the ground up for distributed processing of big data. It overcomes Pandas' limitations through:

1.  **Distributed Computing (Scalability):**
    *   PySpark DataFrames are *distributed collections* of data. This means the data is partitioned across multiple nodes (machines) in a cluster. Each node processes a subset of the data in parallel.
    *   **Advantage for Big Data:** You can process datasets much larger than the memory of any single machine by distributing the workload across a cluster. The cluster's combined memory and processing power can handle petabytes of data.

2.  **In-Memory Processing for Speed (when possible):**
    *   Spark aims to keep data in memory across nodes for intermediate computations, significantly speeding up iterative algorithms and complex queries compared to disk-based systems.

3.  **Lazy Evaluation and Optimized Execution Plan:**
    *   As discussed, PySpark transformations are lazy. Spark builds an optimized logical and physical execution plan (DAG - Directed Acyclic Graph) before running any computations. This optimizer finds the most efficient way to execute your operations across the cluster, minimizing data shuffle and maximizing parallelism.

4.  **Fault Tolerance:**
    *   Spark's core data structure, RDDs (and DataFrames built on top of them), are *Resilient Distributed Datasets*. They can automatically recover from node failures. If a node fails during computation, Spark can recompute the lost partitions from the lineage graph without restarting the entire job.

5.  **Unified Analytics Engine:**
    *   Spark provides a unified platform for various big data tasks: batch processing, stream processing, SQL queries, machine learning (MLlib), and graph processing (GraphX). PySpark makes all these capabilities accessible via Python.

In essence, while Pandas is excellent for smaller datasets and quick, exploratory analysis on a single machine, PySpark is indispensable when your data scales beyond single-machine capabilities, offering parallelism, scalability, and fault tolerance crucial for enterprise-level big data analytics.

## Conclusion: Why PySpark is the Right Tool for Large Datasets

This tutorial has guided you through the fundamentals of PySpark, from setting up a `SparkSession` and loading data, to performing essential exploratory analysis and data transformations. We also conducted a side-by-side comparison with Pandas, highlighting the syntactic similarities and underlying conceptual differences.

The key takeaway is clear:

*   **For small to medium datasets** that fit comfortably in memory, **Pandas** offers a highly intuitive and performant API for data manipulation.
*   **For large to massive datasets (Big Data)** that exceed the memory capacity of a single machine, or when processing speed on large volumes of data is critical, **PySpark** is the superior choice. Its distributed nature, lazy evaluation, and robust fault tolerance mechanisms enable you to process, analyze, and gain insights from data at scales impossible with single-node tools.

By understanding both PySpark and Pandas, you are now equipped to choose the right tool for your data analysis tasks, ensuring efficiency and scalability whether you're working with megabytes or petabytes of data.

## Limitations of Pandas with Big Data and Why PySpark Succeeds

While Pandas is an incredibly powerful and user-friendly library for data manipulation in Python, it faces significant limitations when dealing with datasets that exceed the memory capacity of a single machine. This is where PySpark truly shines.

### Limitations of Pandas:

1.  **Memory Constraints (Single-Machine Execution):**
    *   Pandas DataFrames are *in-memory* objects. This means the entire dataset must fit into the RAM of the machine running the Pandas process. For datasets ranging from gigabytes to terabytes, this becomes a severe bottleneck, leading to `MemoryError` or extremely slow operations due to excessive swapping to disk.
    *   **Implication for Big Data:** You cannot process datasets larger than your machine's available RAM.

2.  **Lack of Parallelism and Distributed Computing:**
    *   Pandas operations are executed on a *single CPU core* by default. While some operations can leverage multiple cores on a single machine (e.g., using `apply` with multiprocessing), it's not inherently designed for distributed computation across a cluster of machines.
    *   **Implication for Big Data:** Processing large datasets sequentially on a single core is prohibitively slow, even if they fit in memory.

3.  **No Built-in Fault Tolerance:**
    *   If a Pandas operation fails midway through a long computation (e.g., due to an unexpected error or system crash), you typically lose all progress and have to restart from the beginning.

### Why PySpark Succeeds:

PySpark is built on Apache Spark, which was designed from the ground up for distributed processing of big data. It overcomes Pandas' limitations through:

1.  **Distributed Computing (Scalability):**
    *   PySpark DataFrames are *distributed collections* of data. This means the data is partitioned across multiple nodes (machines) in a cluster. Each node processes a subset of the data in parallel.
    *   **Advantage for Big Data:** You can process datasets much larger than the memory of any single machine by distributing the workload across a cluster. The cluster's combined memory and processing power can handle petabytes of data.

2.  **In-Memory Processing for Speed (when possible):**
    *   Spark aims to keep data in memory across nodes for intermediate computations, significantly speeding up iterative algorithms and complex queries compared to disk-based systems.

3.  **Lazy Evaluation and Optimized Execution Plan:**
    *   As discussed, PySpark transformations are lazy. Spark builds an optimized logical and physical execution plan (DAG - Directed Acyclic Graph) before running any computations. This optimizer finds the most efficient way to execute your operations across the cluster, minimizing data shuffle and maximizing parallelism.

4.  **Fault Tolerance:**
    *   Spark's core data structure, RDDs (and DataFrames built on top of them), are *Resilient Distributed Datasets*. They can automatically recover from node failures. If a node fails during computation, Spark can recompute the lost partitions from the lineage graph without restarting the entire job.

5.  **Unified Analytics Engine:**
    *   Spark provides a unified platform for various big data tasks: batch processing, stream processing, SQL queries, machine learning (MLlib), and graph processing (GraphX). PySpark makes all these capabilities accessible via Python.

In essence, while Pandas is excellent for smaller datasets and quick, exploratory analysis on a single machine, PySpark is indispensable when your data scales beyond single-machine capabilities, offering parallelism, scalability, and fault tolerance crucial for enterprise-level big data analytics.

## Conclusion: Why PySpark is the Right Tool for Large Datasets

This tutorial has guided you through the fundamentals of PySpark, from setting up a `SparkSession` and loading data, to performing essential exploratory analysis and data transformations. We also conducted a side-by-side comparison with Pandas, highlighting the syntactic similarities and underlying conceptual differences.

The key takeaway is clear:

*   **For small to medium datasets** that fit comfortably in memory, **Pandas** offers a highly intuitive and performant API for data manipulation.
*   **For large to massive datasets (Big Data)** that exceed the memory capacity of a single machine, or when processing speed on large volumes of data is critical, **PySpark** is the superior choice. Its distributed nature, lazy evaluation, and robust fault tolerance mechanisms enable you to process, analyze, and gain insights from data at scales impossible with single-node tools.

By understanding both PySpark and Pandas, you are now equipped to choose the right tool for your data analysis tasks, ensuring efficiency and scalability whether you're working with megabytes or petabytes of data.